In [ ]:
# V4 Cell 1 — Court Geometry

import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Court dimensions (meters)
# -------------------------------------------------------

COURT_WIDTH = 8.23
SERVICE_BOX_WIDTH = COURT_WIDTH / 2

NET_X = 0.0
SERVICE_LINE_X = 6.40
BASELINE_X = 11.885

NET_HEIGHT_CENTER = 0.914
NET_HEIGHT_POST = 1.07

# Lateral boundaries
COURT_HALF_WIDTH = COURT_WIDTH / 2

# Service-box boundaries relative to center service line
SERVICE_CENTERLINE_Y = 0.0

print("=" * 60)
print("V4 — COURT GEOMETRY")
print("=" * 60)

print("\nLongitudinal geometry")
print("-" * 60)
print(f"Net position:             x = {NET_X:.3f} m")
print(f"Service line:             x = {SERVICE_LINE_X:.3f} m")
print(f"Baseline:                 x = {BASELINE_X:.3f} m")

print("\nLateral geometry")
print("-" * 60)
print(f"Singles court width:      {COURT_WIDTH:.3f} m")
print(f"Service-box width:        {SERVICE_BOX_WIDTH:.3f} m")
print(f"Center service line:      y = {SERVICE_CENTERLINE_Y:.3f} m")

print("\nNet geometry")
print("-" * 60)
print(f"Net center height:        {NET_HEIGHT_CENTER:.3f} m")
print(f"Net post height:          {NET_HEIGHT_POST:.3f} m")

print("\nV4 COURT GEOMETRY: DEFINED")

In [ ]:
# V4 Cell 2 — Service Box Geometry

fig, ax = plt.subplots(figsize=(10, 6))

# Court boundaries
ax.plot(
    [-COURT_HALF_WIDTH, COURT_HALF_WIDTH],
    [0, 0],
    linewidth=2,
    label="Net"
)

# Service line
ax.plot(
    [-COURT_HALF_WIDTH, COURT_HALF_WIDTH],
    [SERVICE_LINE_X, SERVICE_LINE_X],
    linewidth=2,
    label="Service line"
)

# Center service line
ax.plot(
    [0, 0],
    [NET_X, SERVICE_LINE_X],
    linestyle="--",
    linewidth=1.5,
    label="Center service line"
)

# Singles sidelines
ax.plot(
    [-COURT_HALF_WIDTH, -COURT_HALF_WIDTH],
    [NET_X, SERVICE_LINE_X],
    linewidth=2
)

ax.plot(
    [COURT_HALF_WIDTH, COURT_HALF_WIDTH],
    [NET_X, SERVICE_LINE_X],
    linewidth=2
)

# Highlight one service box
ax.fill_between(
    [0, SERVICE_BOX_WIDTH],
    NET_X,
    SERVICE_LINE_X,
    alpha=0.2,
    label="Target service box"
)

ax.set_xlabel("Lateral position y (m)")
ax.set_ylabel("Distance from net x (m)")
ax.set_title("Tennis Service Box — Model Coordinate System")

ax.set_xlim(-COURT_HALF_WIDTH - 0.5, COURT_HALF_WIDTH + 0.5)
ax.set_ylim(-0.5, SERVICE_LINE_X + 0.5)

ax.set_aspect("equal")
ax.grid(True)
ax.legend()

plt.show()

In [ ]:
# V4 Cell 3 — Corrected Serve Constraints

def net_height(y):
    """
    Approximate tennis-net height as a function of
    lateral position.

    Height varies from 0.914 m at the center
    toward 1.07 m at the singles sideline.

    This is a provisional linear approximation.
    """

    y = np.asarray(y)

    return (
        NET_HEIGHT_CENTER
        + (NET_HEIGHT_POST - NET_HEIGHT_CENTER)
        * np.minimum(
            np.abs(y) / SERVICE_BOX_WIDTH,
            1.0
        )
    )


def is_inside_service_box(y, target_side="deuce"):
    """
    Determine whether a landing point lies inside
    the selected singles service box.
    """

    if target_side == "deuce":

        return (
            0.0 <= y <= SERVICE_BOX_WIDTH
        )

    elif target_side == "ad":

        return (
            -SERVICE_BOX_WIDTH <= y <= 0.0
        )

    else:
        raise ValueError(
            "target_side must be 'deuce' or 'ad'"
        )


def net_clearance(z, y):
    """
    Vertical clearance above the net.

    Positive  -> above net
    Zero      -> exactly at net height
    Negative  -> below net
    """

    return z - net_height(y)


def is_inside_service_depth(x):
    """
    Determine whether the landing point is inside
    the longitudinal service-box region.

    The legal service area lies BETWEEN the net
    and the service line.

        net          service line
         x=0           x=6.40
          |--------------|
             LEGAL

    """

    return (
        NET_X < x < SERVICE_LINE_X
    )


def serve_is_legal(
    landing_x,
    landing_y,
    net_z,
    net_y,
    target_side="deuce"
):
    """
    Determine whether a serve satisfies the basic
    geometric constraints.

    Conditions:
        1. Ball clears the net.
        2. Ball lands between the net and service line.
        3. Ball lands inside the selected service box.
    """

    clears_net = (
        net_clearance(net_z, net_y) > 0.0
    )

    inside_depth = (
        is_inside_service_depth(landing_x)
    )

    inside_width = (
        is_inside_service_box(
            landing_y,
            target_side=target_side
        )
    )

    return (
        clears_net
        and inside_depth
        and inside_width
    )


print("=" * 60)
print("V4 — CORRECTED SERVE CONSTRAINTS")
print("=" * 60)

print("\nLongitudinal service-box checks")
print("-" * 60)

for x in [-1.0, 0.0, 3.0, 6.4, 7.0]:
    print(
        f"x = {x:5.2f} m  →  "
        f"{'INSIDE' if is_inside_service_depth(x) else 'OUTSIDE'}"
    )

print("\nLateral service-box checks")
print("-" * 60)

for y, side in [
    (2.0, "deuce"),
    (4.115, "deuce"),
    (-2.0, "deuce"),
    (-2.0, "ad"),
    (5.0, "deuce")
]:
    print(
        f"y = {y:6.3f} m, "
        f"target = {side:5s}  →  "
        f"{'INSIDE' if is_inside_service_box(y, side) else 'OUTSIDE'}"
    )

print("\nNet height checks")
print("-" * 60)

for y in [0.0, 2.0, 4.115]:
    print(
        f"y = {y:5.3f} m  →  "
        f"net height = {net_height(y):.4f} m"
    )

print("\nV4 CORRECTED CONSTRAINTS: PASS")

In [ ]:
# V4 Cell 4 — Serve Launch Geometry

# Contact height used for the initial geometric model
CONTACT_HEIGHT = 3.0

# Server is one full court length behind the net
SERVER_X = -BASELINE_X

print("=" * 60)
print("V4 — SERVE LAUNCH GEOMETRY")
print("=" * 60)

print("\nLongitudinal positions")
print("-" * 60)
print(f"Server contact:     x = {SERVER_X:.3f} m")
print(f"Net:                x = {NET_X:.3f} m")
print(f"Service line:       x = {SERVICE_LINE_X:.3f} m")

print("\nDistances from server")
print("-" * 60)
print(
    f"Server → net:       "
    f"{NET_X - SERVER_X:.3f} m"
)

print(
    f"Server → service:   "
    f"{SERVICE_LINE_X - SERVER_X:.3f} m"
)

print("\nServe launch point")
print("-" * 60)
print(
    f"(x, y, z) = "
    f"({SERVER_X:.3f}, 0.000, {CONTACT_HEIGHT:.3f})"
)

print("\nV4 LAUNCH GEOMETRY: DEFINED")

In [ ]:
# V4 Cell 5 — Serve Trajectory Through Court

import numpy as np
from scipy.integrate import solve_ivp

# -------------------------------------------------------
# Physical constants
# -------------------------------------------------------

G = 9.81

BALL_MASS = 0.0575
BALL_DIAMETER = 0.067
BALL_RADIUS = BALL_DIAMETER / 2
BALL_AREA = np.pi * BALL_RADIUS**2

AIR_DENSITY = 1.21
DRAG_COEFFICIENT = 0.55

V_SPIN = 20.0


# -------------------------------------------------------
# Lift coefficient
# -------------------------------------------------------

def lift_coefficient(speed, spin_speed):
    """
    Provisional lift-coefficient model.

    spin_speed = R * |omega| in m/s.

    This model is used for computational testing and
    has not yet been independently calibrated.
    """

    if speed <= 0 or spin_speed <= 0:
        return 0.0

    return 1.0 / (2.0 + speed / spin_speed)


# -------------------------------------------------------
# Magnus acceleration
# -------------------------------------------------------

def magnus_acceleration(
    velocity,
    omega,
    mass=BALL_MASS,
    area=BALL_AREA,
    air_density=AIR_DENSITY
):
    """
    Calculate Magnus acceleration.
    """

    velocity = np.asarray(velocity, dtype=float)
    omega = np.asarray(omega, dtype=float)

    speed = np.linalg.norm(velocity)
    spin_rate = np.linalg.norm(omega)

    if speed == 0 or spin_rate == 0:
        return np.zeros(3)

    velocity_hat = velocity / speed
    omega_hat = omega / spin_rate

    spin_speed = BALL_RADIUS * spin_rate

    C_L = lift_coefficient(
        speed,
        spin_speed
    )

    magnus_direction = np.cross(
        omega_hat,
        velocity_hat
    )

    force_magnitude = (
        0.5
        * air_density
        * area
        * C_L
        * speed**2
    )

    force = (
        force_magnitude
        * magnus_direction
    )

    return force / mass


# -------------------------------------------------------
# Total acceleration
# -------------------------------------------------------

def serve_acceleration(
    velocity,
    omega
):
    """
    Calculate total acceleration from:

        gravity + drag + Magnus
    """

    velocity = np.asarray(
        velocity,
        dtype=float
    )

    speed = np.linalg.norm(velocity)

    # Gravity
    a_gravity = np.array([
        0.0,
        0.0,
        -G
    ])

    # Quadratic drag
    if speed > 0:

        drag_factor = (
            -0.5
            * AIR_DENSITY
            * DRAG_COEFFICIENT
            * BALL_AREA
            * speed
            / BALL_MASS
        )

        a_drag = (
            drag_factor
            * velocity
        )

    else:

        a_drag = np.zeros(3)

    # Magnus
    a_magnus = magnus_acceleration(
        velocity,
        omega
    )

    return (
        a_gravity
        + a_drag
        + a_magnus
    )


# -------------------------------------------------------
# 3D trajectory equation
# -------------------------------------------------------

def serve_trajectory(
    t,
    state,
    omega
):
    """
    3D tennis-ball trajectory.
    """

    x, y, z, vx, vy, vz = state

    velocity = np.array([
        vx,
        vy,
        vz
    ])

    acceleration = serve_acceleration(
        velocity,
        omega
    )

    return np.array([
        vx,
        vy,
        vz,
        acceleration[0],
        acceleration[1],
        acceleration[2]
    ])


# -------------------------------------------------------
# Court events
# -------------------------------------------------------

def net_event(t, state):
    """
    Event when the ball reaches x = 0.
    """

    return state[0] - NET_X


net_event.terminal = False
net_event.direction = 1


def landing_event(t, state):
    """
    Event when the ball reaches z = 0.
    """

    return state[2]


landing_event.terminal = True
landing_event.direction = -1


# -------------------------------------------------------
# Serve simulation
# -------------------------------------------------------

def simulate_serve(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    omega=np.zeros(3)
):
    """
    Simulate a serve from the server's contact point.

    Parameters
    ----------
    speed_kmh : float
        Initial ball speed in km/h.

    launch_angle_deg : float
        Elevation angle above horizontal.

    azimuth_deg : float
        Lateral launch angle in degrees.

    omega : array-like
        Spin vector in rad/s.
    """

    speed = speed_kmh / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )

    # Initial velocity
    vx = (
        speed
        * np.cos(theta)
        * np.cos(phi)
    )

    vy = (
        speed
        * np.cos(theta)
        * np.sin(phi)
    )

    vz = (
        speed
        * np.sin(theta)
    )

    # Initial state
    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT,
        vx,
        vy,
        vz
    ])

    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),

        t_span=(0.0, 4.0),

        y0=initial_state,

        events=[
            net_event,
            landing_event
        ],

        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,

        dense_output=True
    )

    return solution


print("=" * 60)
print("V4 — SERVE TRAJECTORY MODEL")
print("=" * 60)

print("\nTrajectory model:")
print("  Gravity + quadratic drag + Magnus")

print("\nInitial position:")
print(f"  x = {SERVER_X:.3f} m")
print("  y = 0.000 m")
print(f"  z = {CONTACT_HEIGHT:.3f} m")

print("\nNet event:")
print("  x = 0 m")

print("\nLanding event:")
print("  z = 0 m")

print("\nV4 TRAJECTORY MODEL: DEFINED")

In [ ]:
# V4 Cell 6 — Single Serve Test

def extract_serve_events(solution):
    """
    Extract the ball state when it crosses the net
    and when it reaches the court.
    """

    if len(solution.t_events[0]) == 0:
        raise RuntimeError(
            "The trajectory did not cross the net."
        )

    if len(solution.t_events[1]) == 0:
        raise RuntimeError(
            "The trajectory did not reach the court."
        )

    net_state = solution.y_events[0][0]
    landing_state = solution.y_events[1][0]

    return net_state, landing_state


# -------------------------------------------------------
# Simulate test serve
# -------------------------------------------------------

test_solution = simulate_serve(
    speed_kmh=200.0,
    launch_angle_deg=6.0,
    azimuth_deg=10.0,
    omega=np.zeros(3)
)

net_state, landing_state = extract_serve_events(
    test_solution
)


# -------------------------------------------------------
# Display results
# -------------------------------------------------------

print("=" * 60)
print("V4 — SINGLE SERVE TEST")
print("=" * 60)

print("\nLaunch parameters")
print("-" * 60)
print("Speed:          200.0 km/h")
print("Launch angle:     6.0°")
print("Azimuth:         10.0°")
print("Spin:             0 rad/s")

print("\nNet crossing")
print("-" * 60)
print(f"x = {net_state[0]:.6f} m")
print(f"y = {net_state[1]:.6f} m")
print(f"z = {net_state[2]:.6f} m")

print("\nLanding")
print("-" * 60)
print(f"x = {landing_state[0]:.6f} m")
print(f"y = {landing_state[1]:.6f} m")
print(f"z = {landing_state[2]:.6e} m")

print("\nEvent verification")
print("-" * 60)

net_pass = abs(net_state[0] - NET_X) < 1e-8
landing_pass = abs(landing_state[2]) < 1e-8

print(
    f"Net crossing at x = 0: "
    f"{'PASS' if net_pass else 'FAIL'}"
)

print(
    f"Landing at z = 0: "
    f"{'PASS' if landing_pass else 'FAIL'}"
)

print("\nV4 SINGLE-SERVE SIMULATION: PASS")

In [ ]:
# V4 Cell 7 — Expanded Launch Angle Sweep

angles = np.arange(
    -10.0,
    11.0,
    0.5
)

landing_distances = []

for angle in angles:

    solution = simulate_serve(
        speed_kmh=200.0,
        launch_angle_deg=angle,
        azimuth_deg=0.0,
        omega=np.zeros(3)
    )

    try:
        _, landing = extract_serve_events(
            solution
        )

        landing_distances.append(
            landing[0]
        )

    except RuntimeError:
        landing_distances.append(
            np.nan
        )


landing_distances = np.array(
    landing_distances
)


# -------------------------------------------------------
# Plot
# -------------------------------------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    angles,
    landing_distances,
    marker="o"
)

plt.axhspan(
    NET_X,
    SERVICE_LINE_X,
    alpha=0.15,
    label="Service-box depth"
)

plt.axhline(
    NET_X,
    linestyle="--",
    linewidth=1
)

plt.axhline(
    SERVICE_LINE_X,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Launch angle (degrees)")
plt.ylabel("Landing distance from net (m)")

plt.title(
    "Landing Distance vs. Launch Angle — 200 km/h"
)

plt.grid(True)
plt.legend()

plt.show()


# -------------------------------------------------------
# Print useful region
# -------------------------------------------------------

print("=" * 60)
print("V4 — EXPANDED LAUNCH ANGLE SWEEP")
print("=" * 60)

print("\nResults")
print("-" * 60)

for angle, distance in zip(
    angles,
    landing_distances
):

    if np.isfinite(distance):

        print(
            f"{angle:6.1f}°  →  "
            f"landing x = {distance:8.3f} m"
        )

    else:

        print(
            f"{angle:6.1f}°  →  "
            f"no landing detected"
        )


print("\nService-box depth:")
print(
    f"0 < x < {SERVICE_LINE_X:.3f} m"
)

print("\nV4 EXPANDED ANGLE SWEEP: COMPLETE")

In [ ]:
# V4 Cell 8 — Net Clearance vs. Launch Angle

net_heights = []
net_clearances = []
landing_distances_v4 = []

for angle in angles:

    solution = simulate_serve(
        speed_kmh=200.0,
        launch_angle_deg=angle,
        azimuth_deg=0.0,
        omega=np.zeros(3)
    )

    net_state, landing_state = extract_serve_events(
        solution
    )

    y_net = net_state[1]
    z_net = net_state[2]

    clearance = net_clearance(
        z_net,
        y_net
    )

    net_heights.append(
        net_height(y_net)
    )

    net_clearances.append(
        clearance
    )

    landing_distances_v4.append(
        landing_state[0]
    )


net_heights = np.array(net_heights)
net_clearances = np.array(net_clearances)
landing_distances_v4 = np.array(landing_distances_v4)


# -------------------------------------------------------
# Plot net clearance
# -------------------------------------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    angles,
    net_clearances,
    marker="o"
)

plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1,
    label="Net-clearance boundary"
)

plt.xlabel("Launch angle (degrees)")
plt.ylabel("Net clearance (m)")

plt.title(
    "Net Clearance vs. Launch Angle — 200 km/h"
)

plt.grid(True)
plt.legend()

plt.show()


# -------------------------------------------------------
# Print results
# -------------------------------------------------------

print("=" * 60)
print("V4 — NET CLEARANCE ANALYSIS")
print("=" * 60)

print("\nLaunch angle → net clearance")
print("-" * 60)

for angle, clearance in zip(
    angles,
    net_clearances
):

    print(
        f"{angle:6.1f}°  →  "
        f"net clearance = {clearance:8.4f} m"
    )


print("\nV4 NET CLEARANCE ANALYSIS: COMPLETE")

In [ ]:
# V4 Cell 9 — Combined Serve Classification

legal_angles = []

print("=" * 60)
print("V4 — COMBINED SERVE CLASSIFICATION")
print("=" * 60)

print("\nAngle-by-angle classification")
print("-" * 60)

for angle, distance, clearance in zip(
    angles,
    landing_distances_v4,
    net_clearances
):

    clears_net = clearance > 0.0

    inside_depth = (
        NET_X < distance < SERVICE_LINE_X
    )

    legal = (
        clears_net
        and inside_depth
    )

    if legal:
        legal_angles.append(angle)

    status = "LEGAL" if legal else "FAULT"

    print(
        f"{angle:6.1f}°  | "
        f"landing = {distance:7.3f} m | "
        f"clearance = {clearance:7.3f} m | "
        f"{status}"
    )


print("\n" + "-" * 60)

if legal_angles:

    print(
        f"Grid-based legal interval: "
        f"{min(legal_angles):.1f}° "
        f"to "
        f"{max(legal_angles):.1f}°"
    )

else:

    print("No legal trajectories found.")

print("\nV4 COMBINED CLASSIFICATION: COMPLETE")

In [ ]:
# V4 Cell 10 — Precise Admissibility Boundaries

from scipy.optimize import brentq


# -------------------------------------------------------
# Net-clearance function
# -------------------------------------------------------

def net_clearance_for_angle(angle_deg):
    """
    Return net clearance for a given launch angle.
    """

    solution = simulate_serve(
        speed_kmh=200.0,
        launch_angle_deg=angle_deg,
        azimuth_deg=0.0,
        omega=np.zeros(3)
    )

    net_state, _ = extract_serve_events(
        solution
    )

    return net_clearance(
        net_state[2],
        net_state[1]
    )


# -------------------------------------------------------
# Landing-distance function
# -------------------------------------------------------

def landing_distance_for_angle(angle_deg):
    """
    Return landing distance from the net.
    """

    solution = simulate_serve(
        speed_kmh=200.0,
        launch_angle_deg=angle_deg,
        azimuth_deg=0.0,
        omega=np.zeros(3)
    )

    _, landing_state = extract_serve_events(
        solution
    )

    return landing_state[0]


# -------------------------------------------------------
# Find lower boundary
# -------------------------------------------------------

theta_net_boundary = brentq(
    net_clearance_for_angle,
    -9.0,
    -8.5
)


# -------------------------------------------------------
# Find upper boundary
# -------------------------------------------------------

theta_service_boundary = brentq(
    lambda angle:
        landing_distance_for_angle(angle)
        - SERVICE_LINE_X,

    -7.5,
    -7.0
)


# -------------------------------------------------------
# Evaluate boundary conditions
# -------------------------------------------------------

net_at_boundary = net_clearance_for_angle(
    theta_net_boundary
)

landing_at_service_boundary = (
    landing_distance_for_angle(
        theta_service_boundary
    )
)


# -------------------------------------------------------
# Results
# -------------------------------------------------------

print("=" * 60)
print("V4 — PRECISE ADMISSIBILITY BOUNDARIES")
print("=" * 60)

print("\nLower boundary — net clearance")
print("-" * 60)
print(
    f"Launch angle = "
    f"{theta_net_boundary:.8f}°"
)

print(
    f"Net clearance = "
    f"{net_at_boundary:.6e} m"
)

print("\nUpper boundary — service line")
print("-" * 60)
print(
    f"Launch angle = "
    f"{theta_service_boundary:.8f}°"
)

print(
    f"Landing distance = "
    f"{landing_at_service_boundary:.8f} m"
)

print("\nAdmissible interval")
print("-" * 60)

print(
    f"{theta_net_boundary:.8f}° "
    f"< θ < "
    f"{theta_service_boundary:.8f}°"
)

print("\nV4 PRECISE BOUNDARY CALCULATION: COMPLETE")

In [ ]:
# V4 Cell 11 — Boundary Verification

theta_lower = theta_net_boundary
theta_upper = theta_service_boundary

# Test points just inside/outside each boundary
test_angles = [
    theta_lower - 0.05,
    theta_lower + 0.05,
    (theta_lower + theta_upper) / 2,
    theta_upper - 0.05,
    theta_upper + 0.05
]

print("=" * 60)
print("V4 — BOUNDARY VERIFICATION")
print("=" * 60)

print("\nAngle classification")
print("-" * 60)

for angle in test_angles:

    solution = simulate_serve(
        speed_kmh=200.0,
        launch_angle_deg=angle,
        azimuth_deg=0.0,
        omega=np.zeros(3)
    )

    net_state, landing_state = extract_serve_events(
        solution
    )

    clearance = net_clearance(
        net_state[2],
        net_state[1]
    )

    landing_x = landing_state[0]

    clears_net = clearance > 0
    inside_depth = (
        NET_X < landing_x < SERVICE_LINE_X
    )

    legal = (
        clears_net
        and inside_depth
    )

    print(
        f"\nθ = {angle: .6f}°"
    )

    print(
        f"  Net clearance = "
        f"{clearance: .6f} m"
    )

    print(
        f"  Landing x     = "
        f"{landing_x: .6f} m"
    )

    print(
        f"  Net: "
        f"{'CLEAR' if clears_net else 'FAULT'}"
    )

    print(
        f"  Depth: "
        f"{'INSIDE' if inside_depth else 'OUTSIDE'}"
    )

    print(
        f"  Classification: "
        f"{'LEGAL' if legal else 'FAULT'}"
    )


print("\n" + "-" * 60)
print(
    f"Admissible interval: "
    f"{theta_lower:.6f}° < θ < {theta_upper:.6f}°"
)

print("\nV4 BOUNDARY VERIFICATION: COMPLETE")

In [ ]:
# V4 Cell 12 — Visualize the 1D Admissible Interval

# Fine angle grid around the admissible region
fine_angles = np.linspace(
    theta_net_boundary - 0.5,
    theta_service_boundary + 0.5,
    200
)

fine_landing = []
fine_clearance = []

for angle in fine_angles:

    solution = simulate_serve(
        speed_kmh=200.0,
        launch_angle_deg=angle,
        azimuth_deg=0.0,
        omega=np.zeros(3)
    )

    net_state, landing_state = extract_serve_events(
        solution
    )

    fine_landing.append(
        landing_state[0]
    )

    fine_clearance.append(
        net_clearance(
            net_state[2],
            net_state[1]
        )
    )

fine_landing = np.array(fine_landing)
fine_clearance = np.array(fine_clearance)


# -------------------------------------------------------
# Determine legal region
# -------------------------------------------------------

legal_mask = (
    (fine_clearance > 0)
    & (fine_landing > NET_X)
    & (fine_landing < SERVICE_LINE_X)
)


# -------------------------------------------------------
# Plot
# -------------------------------------------------------

plt.figure(figsize=(11, 6))

plt.plot(
    fine_angles,
    fine_landing,
    label="Landing distance"
)

plt.axhspan(
    NET_X,
    SERVICE_LINE_X,
    alpha=0.15,
    label="Service-box depth"
)

plt.axvline(
    theta_net_boundary,
    linestyle="--",
    label="Net boundary"
)

plt.axvline(
    theta_service_boundary,
    linestyle="--",
    label="Service-line boundary"
)

plt.scatter(
    fine_angles[legal_mask],
    fine_landing[legal_mask],
    s=10,
    label="Admissible"
)

plt.xlabel("Launch angle θ (degrees)")
plt.ylabel("Landing distance from net (m)")

plt.title(
    "1D Serve Admissibility at 200 km/h"
)

plt.grid(True)
plt.legend()

plt.show()


# -------------------------------------------------------
# Summary
# -------------------------------------------------------

interval_width = (
    theta_service_boundary
    - theta_net_boundary
)

print("=" * 60)
print("V4 — 1D ADMISSIBILITY SUMMARY")
print("=" * 60)

print(
    f"\nNet boundary:       "
    f"{theta_net_boundary:.6f}°"
)

print(
    f"Service boundary:   "
    f"{theta_service_boundary:.6f}°"
)

print(
    f"Admissible width:   "
    f"{interval_width:.6f}°"
)

print(
    f"\nSpeed:              200 km/h"
)

print(
    f"Azimuth:            0°"
)

print(
    f"Spin:               0 rad/s"
)

print("\nV4 1D ADMISSIBILITY: COMPLETE")